# Deterministic Retrieval-Augmented Generation Pipeline
Prerequisites: text normalization and lists. Estimated time: 35 minutes. Objectives: rank a small document set deterministically, assemble bounded evidence context, and handle no-match failure cases; duplicate-document deduplication is an exercise.

## Concept recap and assumptions
A RAG pipeline retrieves evidence from an external corpus and supplies it to a generator. This notebook implements only lexical retrieval and context assembly over synthetic documents. It does not generate answers, measure semantic relevance, or represent a production search system.

In [ ]:
def tokens(text):
    return set(text.lower().split())

def retrieve(query, documents, k=3):
    if k < 0:
        raise ValueError('k must be non-negative')
    scored = []
    for doc_id, text in documents.items():
        score = len(tokens(query) & tokens(text))
        if score:
            scored.append((score, doc_id, text))
    return sorted(scored, key=lambda row: (-row[0], row[1]))[:k]

def assemble_context(query, documents, k=3, max_chars=160):
    hits = retrieve(query, documents, k)
    chunks = []
    for score, doc_id, text in hits:
        item = f'[{doc_id}] {text}'
        if sum(map(len, chunks)) + len(item) > max_chars:
            continue
        chunks.append(item)
    return chunks

corpus = {'policy-a': 'refunds are available within thirty days', 'policy-b': 'damaged items qualify for replacement', 'faq': 'support is available by email'}
query = 'damaged replacement policy'
hits = retrieve(query, corpus, k=2)
context = assemble_context(query, corpus, k=2)
hits

## Baseline assertions
Token overlap is transparent and cheap, but it misses synonyms and can reward common words. Stable document-ID ordering makes ties reproducible. Context assembly retains source IDs so downstream answers can cite their evidence.

In [ ]:
assert [doc_id for _, doc_id, _ in hits] == ['policy-b']
assert context[0].startswith('[policy-b]')
assert all(doc_id in context[0] for doc_id in ('policy-b',))

## Comparison and metric
For a labeled query set, retrieval can be measured with Recall@k or reciprocal rank. The toy query has relevant document `policy-b`; its rank is one. This is a hand-authored example, not a measured benchmark. A real system should inspect retrieved text and evaluate retrieval separately from generation.

In [ ]:
relevant_id = 'policy-b'
rank = next((i for i, (_, doc_id, _) in enumerate(hits, 1) if doc_id == relevant_id), None)
reciprocal_rank = 1 / rank if rank else 0
assert reciprocal_rank == 1.0
# Failure cases: no evidence, zero k, negative k, context budget, and stable ties.
assert retrieve('quantum telescope', corpus) == []
assert assemble_context('quantum telescope', corpus) == []
assert retrieve('policy support', corpus, k=0) == []
try:
    retrieve('policy', corpus, k=-1)
except ValueError:
    pass
else:
    raise AssertionError('negative k must fail')
assert assemble_context('damaged replacement policy', corpus, max_chars=5) == []
assert retrieve('policy', {'z': 'policy text', 'a': 'policy guide'})[0][1] == 'a'

## Exercises
1. Replace set overlap with term frequency and explain how repeated query terms should count. 2. Add a minimum score and make the pipeline abstain when no chunk qualifies. 3. Add duplicate-content documents and deduplicate while preserving the preferred source. Expected properties: retrieval remains deterministic, context respects its character budget, and no-match returns no fabricated evidence.

## Summary, next lesson, and practice
The pipeline ranks synthetic documents, assembles bounded source-labeled context, and exposes failure cases. Lexical overlap is not semantic retrieval, and context alone does not guarantee a correct generated answer. Next: vector databases and semantic search. Practice: [RAG interview drill](../interview-prep/curriculum-practice.md#rag-retrieval-augmented-generation).